# Chapter 4 &mdash; The Language of a DFA, and "Regular"

**Concept 7 of the Chapter 4 decomposition:** *The Language of a DFA, and the Definition of a Regular Language*

<b>Accept</b> is for strings, <b>recognize</b> is for languages. A regular language is the language of <i>some</i> DFA.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Language-Of-A-DFA/Concept-Language-Of-A-DFA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The **language of a DFA** is the set of strings it accepts; the DFA **recognizes** that
language.

The vocabulary is used precisely from here on: **accept** for single *strings*,
**recognize** for *languages*.

> **A regular language over $\Sigma$ is the language of some DFA with this alphabet.**

Note the shape: that is **existential over machines**. It is why proving a language
**non**-regular is hard &mdash; you must rule out *every* machine &mdash; and why the Pumping
Lemma exists.

## 2. Definitions

### Enumerating the language of a DFA, up to a length

In [ ]:
from itertools import product

def language_upto(D, n):
    out = []
    for k in range(n+1):
        for p in product(sorted(D["Sigma"]), repeat=k):
            s = ''.join(p)
            if accepts_dfa(D, s):
                out.append(s)
    return out

### Two machines for the same language

In [ ]:
odd1_a = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> F
F : 1 -> I
''')
# NOTE the naming rule: only ONE state may start with 'I'.  The extra
# non-final state is called A, not I2 -- 'I2' would make a second start state.
odd1_b = md2mc('''DFA
I : 0 -> A
I : 1 -> F
A : 0 -> I
A : 1 -> F
F : 0 -> F
F : 1 -> I
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;6.&nbsp;A DFA as a Goto-Based Program](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-DFA-As-Goto-Program/Concept-DFA-As-Goto-Program.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;8.&nbsp;DFA as String Classifiers: Partitioning $\Sigma^](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-DFA-As-String-Classifier/Concept-DFA-As-String-Classifier.ipynb)&nbsp;&rarr;

---

## 3. Tests

The language of the first machine: odd number of 1s.

In [ ]:
L = language_upto(odd1_a, 4)
print("accepted up to length 4 :", L[:12], "...")
assert all(s.count('1') % 2 == 1 for s in L)
print("every accepted string has an odd number of 1s :", True)

**Two different machines, one language.** "Regular" is existential over machines, so both witness the same fact.

In [ ]:
print("machine A has %d states, machine B has %d" % (len(odd1_a["Q"]), len(odd1_b["Q"])))
print("same language? ", langeq_dfa(odd1_a, odd1_b))
print("isomorphic?    ", iso_dfa(odd1_a, odd1_b))
assert langeq_dfa(odd1_a, odd1_b) and not iso_dfa(odd1_a, odd1_b)
print("\nSame language, different structure -- 'regular' asks only that SOME DFA exists.")

Historically, regular languages were defined by **regular expressions**; Kleene's theorem connected the two. (Chapters 8-10.)

In [ ]:
print("DFA-based definition : L is regular iff SOME DFA recognizes it")
print("RE-based definition  : L is regular iff SOME RE denotes it")
print("Kleene's theorem     : the two coincide")

## 4. Exercises


1. Give three more DFA for "odd number of 1s". How many are there in total?
2. Why is "*some* DFA exists" harder to refute than "*this* DFA works"?
3. Write down the language of the two-state even-0s machine as a set comprehension.

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4/Concept-Language-Of-A-DFA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')